### Types of Partitioning in Spark
<ul>
    Partition is generally used when there is a high cardinality
    <li>Hash Partitioning</li>
    <li>Range Partitioning</li>
</ul>

#### Hash Partitioning
<ul>
    <li>Hash Partitioning attempts to spread the data evenly across various partitions based
on the key.</li>
    <li> Object.hashCode method is used to determine the partition in Spark as
partition = key.hashCode ( ) % numPartitions.</li>
</ul>

#### Range Partitioning
<ul>
    <li>Some Spark RDDs have keys that follow a particular ordering for such RDDs range
partitioning is an efficient partitioning technique.</li>
    <li>In range partitioning method, tuples having keys within the same range will appear on
the same machine.</li>
    <li>Keys in a range partitioner are partitioned based on the set of sorted range of keys and
ordering of keys.</li>
</ul>

### When are we supposed to Use Partitioning and bucketting 

#### When to Use Partitioning

- You frequently **filter** by a column (e.g., `date`, `region`)
- The column has **low to medium cardinality**
- You want to **reduce scan size** and leverage **partition pruning**
- You're storing data in **S3 or HDFS**, where folder-based layout helps
- Your queries are **time-based**, like daily or monthly reports


#### When to Use Bucketing

- You frequently **join** or **groupBy** on a column (e.g., `user_ID`, `session_ID`)
- The column has **high cardinality**
- You want to **avoid shuffle** during joins or aggregations
- Both sides of a join are bucketed on the same column
- You're writing to a **Hive-compatible table** using `.saveAsTable()`


| Feature         | Partitioning                            | Bucketing                                 |
|----------------|------------------------------------------|-------------------------------------------|
| Storage Layout | Creates folders per partition column     | Stores bucketed files inside table folder |
| Optimization   | Enables partition pruning                | Enables shuffle avoidance and bucket pruning |
| Column Type    | Best for low-cardinality columns         | Best for high-cardinality columns         |
| Use Case       | Filter-heavy queries                     | Join-heavy or groupBy-heavy queries       |
| Compatibility  | Works well with S3/HDFS                  | Requires Hive-compatible table metadata   |

### Delta Table

<p>A Delta table is a data management feature provided by Delta Lake, an open-source storage layer that brings ACID transactions, schema enforcement, and time travel to big data workloads—especially in Spark.</p>

<p>Delta table originated at Databricks as a key component of Delta Lake, an open-source storage layer for Apache Spark developed to bring ACID transactions, versioned data, and reliability to data lakes. Databricks created the Delta Lake protocol and continues to contribute to the open-source project, with Delta tables serving as the foundation for the lakehouse architecture. They provide a unified system for batch and streaming data, enabling operations like UPDATE, DELETE, and INSERT that were difficult with traditional data lakes.  </p>

<table style="text-align: left;font-size: 2.25ex">
    <tr >
        <th>Feature</th>
        <th>Benefit</th>
    </tr>
    <tr>
        <th>ACID transactions</th>
        <td>Guarantees atomicity, consistency, isolation, durability—even in distributed writes</td>
    </tr>
    <tr>
        <th>Schema enforcement & evolution</th>
        <td>Prevents bad data from corrupting tables; allows controlled schema changes</td>
    </tr>
    <tr>
        <th>Time travel</th>
        <td>Query historical versions of data using `versionAsOf` or `timestampAsOf`</td>
    </tr>
    <tr>
        <th>Upserts & deletes</th>
        <td>Supports `MERGE`, `UPDATE`, `DELETE—unlike` standard Parquet
</td>
    </tr>
    <tr>
        <th>Auto compaction & Z-ordering</th>
        <td>Optimizes read performance and file layout
</td>
    </tr>
    <tr>
        <th>Streaming support</th>
        <td>Seamless integration with Structured Streaming for both read/write</td>
    </tr>
</table>    


### Explain how Delta Lake transactions work

<p>
Delta Lake transactions work by maintaining <b>an atomic transaction log that records all changes to a table as ordered JSON files. Each file, or commit, contains metadata about the changes, such as which data files were added or removed.</b> When a write operation occurs, the new data files are written first, and then a new JSON commit file is written to the transaction log, making the changes visible and providing a consistent, atomic state of the table. This log enables features like data versioning, time travel, and ACID compliance, ensuring data reliability by preventing partial reads or duplicate data. 
</p>

In [1]:
# example of Delta table 
# loading data from s3 to delta configuring all those things
# 

from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from delta.tables import DeltaTable

schema = StructType([
    StructField("record_id", StringType(), True),
    StructField("category", StringType(), True),
    StructField("score", DoubleType(), True)
])

# Load Data Using the Schema
new_df = spark.read.schema(schema).json("s3://incoming-data/2025-09-28/")

# Create New Delta Table
new_df.write.format("delta").mode("overwrite").save("s3://your-bucket/delta/category_scores")

# Append to Existing Delta Table
new_df.write.format("delta").mode("append").save("s3://your-bucket/delta/category_scores")

# Register as a Table (Optional)
spark.sql("""
CREATE TABLE IF NOT EXISTS category_scores
USING DELTA
LOCATION 's3://your-bucket/delta/category_scores'
""")

In [ ]:
# Maintain with MERGE (Upsert Logic)
delta_table = DeltaTable.forPath(spark, "s3://your-bucket/delta/category_scores")

delta_table.alias("target").merge(
    new_df.alias("source"),
    "target.record_id = source.record_id"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()


In [ ]:
from pyspark.conf import SparkConf
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from delta.tables import DeltaTable

conf = SparkConf().setAppName("185")\ 
                  .setMaster("local[4]")

conf.set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
conf.set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")


spark = SparkSession.builder.config(conf = conf).getOrCreate()

# Define schema
schema = StructType([
    StructField("record_id", StringType(), True),
    StructField("category", StringType(), True),
    StructField("score", IntegerType(), True)
])


# Set partition date
partition_date = "20250928"
source_path = f"s3://incoming-data/{partition_date}/"


spark.sql("""
CREATE TABLE IF NOT EXISTS category_scores
USING DELTA
PARTITIONED BY (file_date_partition)
LOCATION 's3://your-bucket/delta/category_scores'
""")


# Load data and add partition column
df = spark.read.schema(schema).json(source_path) \
    .withColumn("file_date_partition", F.lit(partition_date))

# Simple append the data to delta table source
df.write.format("delta") \
    .partitionBy("file_date_partition") \
    .mode("append") \
    .save("s3://your-bucket/delta/category_scores")


## SCD1 of Delta tables

In [ ]:
from delta.tables import DeltaTable
import pyspark.sql.functions as F

# Load new data
new_df = spark.read.json("s3://incoming-data/20250928/") \
    .withColumn("file_date_partition", F.lit("20250928"))

# Load target Delta table
delta_table = DeltaTable.forPath(spark, "s3://your-bucket/delta/category_scores")

# Merge: overwrite existing records
delta_table.alias("target").merge(
    new_df.alias("source"),
    "target.record_id = source.record_id"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()


## SCD2 of Delta Tables

In [ ]:
## For SCD2 better have a current_flag to identify the latest version of that record

from delta.tables import DeltaTable

# Load new data
new_df = spark.read.json("s3://incoming-data/20250928/") \
                    .withColumn("file_date_partition", F.lit("20250928")) \
                    .withColumn("is_current", F.lit(True)) 


# Load target Delta table
delta_table = DeltaTable.forPath(spark, "s3://your-bucket/delta/category_scores")


# Expire matching current records
delta_table.alias("target").merge(
    new_df.alias("source"),
    "target.record_id = source.record_id AND target.is_current = True"
).whenMatchedUpdate(set={
    "is_current": F.lit(False)
}).execute()


# Insert new version
delta_table.alias("target").merge(
    new_df.alias("source"),
    "target.record_id = source.record_id AND target.is_current = False"
).whenNotMatchedInsertAll().execute()

## Melwin Important Note :
## we cannot use the join contition like F.col("target.record_id") = F.col("source.record_id") 
## for delta lake contiton it will fail

𝐎𝐩𝐭𝐢𝐦𝐢𝐳𝐚𝐭𝐢𝐨𝐧 𝐭𝐞𝐜𝐡𝐧𝐢𝐪𝐮𝐞𝐬 𝐟𝐨𝐫 𝐝𝐞𝐥𝐭𝐚 𝐭𝐚𝐛𝐥𝐞𝐬

It's not enough to just use Delta; you need to optimize it. Understanding these techniques is crucial for building performant <br>and cost-effective data pipelines.
<br>
1. VACUUM Operations<br>
Deletes data files from the table directory that are no longer referenced by the Delta table's transaction log. This is <br>essential for freeing up storage space and preventing accidental queries on old data.
<pre>Example: VACUUM events RETAIN 168 HOURS;</pre><br>
<br>
2. OPTIMIZE Operation<br>
Compacts a large number of small data files into a smaller number of larger, more efficient files. This dramatically improves <br>query performance by reducing the number of file reads and overhead.
<pre>Example: OPTIMIZE my_transactions_table;</pre><br>
<br>
3. Z-ORDERING<br>
A technique that physically co-locates related data on disk based on one or more columns. It’s highly effective for speeding up <br>queries with multi-column filters.
<pre>Example: OPTIMIZE my_transactions_table ZORDER BY (user_id, event_type);</pre><br>
<br>
4. Partition Pruning<br>
A fundamental optimization where Spark automatically skips entire directories of data that don't satisfy the query's filter <br>conditions. This drastically reduces the amount of data scanned.
<pre>Example: SELECT * FROM sales WHERE date > '2025-09-01'; (Spark automatically prunes all partitions for dates before this one).</pre><br>
